In [20]:
# Upload file
from google.colab import files
uploaded = files.upload()

# Read CSV
import pandas as pd

df = pd.read_csv('healthcare_data_cleaning_dataset.csv')

# Display data
df.head()

Saving healthcare_data_cleaning_dataset.csv to healthcare_data_cleaning_dataset (2).csv


,Patient_ID,Age,Gender,City,Diagnosis,Hospital_Visits,Treatment_Cost,Insurance_Coverage,Admission_Date
0,17270,35.0,Male,Bangalore,Hypertension,13,41010.0,1,2023-11-30
1,10860,21.0,Female,Hyderabad,Flu,11,12194.0,1,2023-02-23
2,15390,77.0,Female,Bangalore,Asthma,2,45086.0,0,2023-03-14
3,15191,79.0,Female,Mumbai,Asthma,13,40842.0,0,2023-08-01
4,15734,60.0,Female,Delhi,Asthma,1,9873.0,1,2023-06-20


Q1. Missing Data Identification

Scenario:
 The hospital suspects incomplete patient records.

Task:

Identify missing values in each column

Calculate percentage of missing data



Objective

Identify missing values in each column and calculate the percentage of missing data.

In [21]:
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

print(missing_values)
print(missing_percentage)

Patient_ID              0
Age                   600
Gender                  0
City                    0
Diagnosis               0
Hospital_Visits         0
Treatment_Cost        593
Insurance_Coverage      0
Admission_Date          0
dtype: int64
Patient_ID             0.000000
Age                   11.764706
Gender                 0.000000
City                   0.000000
Diagnosis              0.000000
Hospital_Visits        0.000000
Treatment_Cost        11.627451
Insurance_Coverage     0.000000
Admission_Date         0.000000
dtype: float64


Missing Values Summary
Column:Missing Values	Percentage (%)
Patient_ID	  0	           0.00
Age	         600	         11.76
Gender	      0	           0.00
City	        0	           0.00
Diagnosis	    0	           0.00
Hospital_Visits	0	         0.00
Treatment_Cost	593	       11.63
Insurance_Coverage	0	     0.00
Admission_Date	0	         0.00
Conclusion
Missing values are present only in:
Age
Treatment_Cost
All other columns are complete.

Q2. Handling Missing Age

Scenario:

 Age is critical for medical analysis, but some values are missing.

Task:

Replace missing Age values with an appropriate method

Justify your choice (mean/median)

Scenario

Age is an important feature for healthcare analysis.

Statistical Analysis

Mean Age ≈ 49.60
Median Age = 50

Chosen Method

Median Imputation

Why Median?

Median is preferred because:

Age can contain extreme values.

Median is resistant to outliers.

It preserves the central tendency better than mean when data may not be perfectly symmetric.

In [22]:
median_age = df['Age'].median()

df['Age'] = df['Age'].fillna(median_age)

Result

All missing age values were replaced with:50

Q3. Handling Missing Treatment Cost

Scenario:
 Treatment cost is highly skewed due to expensive treatments.

Task:

Handle missing Treatment_Cost values

Choose the correct imputation method and explain why

Scenario

Treatment cost is highly skewed because some treatments are extremely expensive.

Statistical Observation

Treatment_Cost Skewness ≈ 4.49

This indicates a heavily right-skewed distribution.

Chosen Method

Median Imputation

Why Median Instead of Mean?

Mean is heavily affected by extremely expensive treatments.

Median is robust against outliers.

Median better represents the typical treatment cost.

In [23]:
median_cost = df['Treatment_Cost'].median()

df['Treatment_Cost'] = df['Treatment_Cost'].fillna(median_cost)

Result

Missing treatment costs were replaced using the median value.

Q4. Duplicate Patient Records

Scenario:
 Some patient records were entered multiple times.

Task:

Identify duplicate rows

Remove duplicates

Compare dataset size before and after

Objective

Identify duplicate records and remove them.

In [24]:
# Count duplicate rows
duplicate_rows = df.duplicated().sum()

# Remove duplicates
df_cleaned = df.drop_duplicates()

Results
Description	                   Count
Dataset Size Before Removing Duplicates	                    5100
Duplicate Rows Found	         99
Dataset Size After Removing Duplicates	                   5001

Conclusion

99 duplicate records were identified and removed.

Removing duplicates improves data quality and avoids repeated patient analysis.

Q5. Invalid Age Values (Data Quality Check)

Scenario:
 Some patients have unrealistic age values (e.g., >100 or <0).

Task:

Detect such records

Decide whether to remove or correct them

Scenario

Check for unrealistic age values such as:

Age < 0
Age > 100

In [25]:
invalid_age = df[(df['Age'] < 0) | (df['Age'] > 100)]

Result

No invalid age records were found.

Conclusion

The dataset does not contain unrealistic age values.

No correction or removal was required.

Q6. Outlier Detection (Treatment Cost)

Scenario:
 Extreme treatment costs are affecting analysis.

Task:

Detect outliers using IQR method

Display number of outliers

Objective

Detect extreme treatment costs using the IQR method.

IQR Formula

Q1 = 25th Percentile

Q3 = 75th Percentile

IQR = Q3 − Q1

Outlier Conditions:

Lower Bound = Q1 − 1.5 × IQR

Upper Bound = Q3 + 1.5 × IQR

In [26]:
Q1 = df['Treatment_Cost'].quantile(0.25)
Q3 = df['Treatment_Cost'].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df['Treatment_Cost'] < lower_bound) |
    (df['Treatment_Cost'] > upper_bound)
]

Calculated Values
Metric	           Value
Q1	               12498
Q3	               37922
IQR	               25424
Lower Bound	      -25638
Upper Bound	       76058
Number of Outliers 50

Conclusion

50 records were identified as outliers.

These are unusually expensive treatments.

Q7. Outlier Treatment

Scenario:
The business team wants to retain all records.

Task:

Apply capping (Winsorization) on Treatment_Cost

Use 5th and 95th percentile



Scenario

The business team wants to retain all records instead of deleting outliers.

Chosen Technique

Winsorization (Capping)

Method Used

Lower Cap = 5th Percentile

Upper Cap = 95th Percentile

Percentile Values
Percentile	         Value
5th Percentile	     3299.45
95th Percentile	     47918

In [28]:
lower_cap = df['Treatment_Cost'].quantile(0.05)
upper_cap = df['Treatment_Cost'].quantile(0.95)

# Apply capping

df['Treatment_Cost_Capped'] = df['Treatment_Cost'].clip(
    lower=lower_cap,
    upper=upper_cap)


Benefits

Retains all patient records.

Reduces the influence of extreme treatment costs.

Improves model stability and statistical analysis.

Q8. Transformation

Scenario:
 Treatment cost is highly skewed.

Task:

Apply log transformation

Create a new column

Compare before vs after distribution

Scenario

Treatment cost distribution is highly skewed.

Objective

Apply log transformation to reduce skewness.

In [29]:
import numpy as np

# Log Transformation

df['Log_Treatment_Cost'] = np.log1p(df['Treatment_Cost'])

Distribution Comparison

Metric	Before Log Transform	After Log Transform

Skewness	 4.80	  -1.26

Interpretation

Before transformation:

Distribution was heavily right-skewed.

After transformation:

Distribution became significantly more balanced.

Extreme values were compressed.

Benefits of Log Transformation

Improves normality.

Reduces effect of outliers.

Helps machine learning models perform better.

Makes statistical analysis more reliable.

Q9. Time-Based Missing Handling

Scenario:
 Admission dates should follow a logical sequence.

Task:

Sort data by Admission_Date

Apply forward fill or backward fill where appropriate

Justify your choice

Scenario

Admission dates should follow a logical sequence.

Objective

Sort records by Admission_Date and apply forward fill/backward fill if required.

Observation
No missing values were found in Admission_Date.
Therefore, no filling was necessary.

In [30]:
# Convert to datetime

df['Admission_Date'] = pd.to_datetime(df['Admission_Date'])

# Sort data

df = df.sort_values(by='Admission_Date')

# Example of forward fill (if needed)

df['Admission_Date'] = df['Admission_Date'].fillna(method='ffill')

/tmp/ipykernel_2746/516858452.py:11: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['Admission_Date'] = df['Admission_Date'].fillna(method='ffill')


Why Forward Fill?

Forward fill is preferred for time-series or chronological healthcare data because:

It preserves sequence continuity.

It uses the most recent valid observation.

It avoids introducing unrealistic future values.

Conclusion

Dataset was successfully sorted by admission date.

No missing dates existed, so no imputation was required.